# 2. Data and Clear Data
In the previous notebook, "first analysis data," we examined the data as follows:  
1. Data type  
2. Data issues and problems  
3. Comparing data with each other  
4. Plotting charts from the data  
5. Assessing the data distribution to determine if it is normal or not  
6. Analyzing data behavior before movement, during movement, and after movement  
7. Identifying the type of movement and the dominant foot (which foot bears the most force and pulls the movement toward itself)  
8. Examining movement behavior  

In [1]:
#basic
import os
import math

#analysis
import numpy as np
import pandas as pd

#Statistik
import scipy.stats as stats

#plot
import matplotlib.pyplot as plt
import seaborn as sns

one file for sampel

In [5]:
# file_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-11 09-38-11-074_Asphalt.txt"
file_path =  r"C:/Users/user/Desktop/Fatemeh/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"

with open(file_path, 'r', encoding='utf-8') as f:
    first_two_lines = [next(f) for _ in range(2)]

for i, line in enumerate(first_two_lines, 1):
    print(f"line {i}: {line.rstrip()}")

# name file
print(first_two_lines[0][5:].replace(".pdo\n",""))
# name Type of surface and last number
print(first_two_lines[1][8:].replace("\n","").split("_"))

line 1: File: loadsol_25-11-17 10-09-36-341.pdo
line 2: Comment:S07_Asphalt_01
 loadsol_25-11-17 10-09-36-341
['S07', 'Asphalt', '01']


# Force Data of Foot

In [23]:
# file_path =  r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"
file_path =  r"C:/Users/user/Desktop/Fatemeh/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-17 10-09-36-341.txt"

def make_dataframe(path):
    """find data in text
    Data = read file
    path = path of file
    line1 = 1st line name of file
    line2 = 2st line name of person and surface and one number
    line3 = 3rd line name of columns
    line4 = 4th line name of columns
    """
    #read data and line 1,2 ,3 4
    with open(path, "r", encoding="utf-8") as f:
        line1 = f.readline().rstrip("\n")
        line2 = f.readline().rstrip("\n")
        line3 = f.readline().rstrip("\n")
        line4 = f.readline().rstrip("\n")
    # line 1 and 2 and make name of columns
    a1 = np.array(line3.split("\t"))
    a2 = np.array(line4.split("\t"))[0:len(a1)]
    b = a1 + a2
    # replace name of columns
    name_colunms = np.char.replace(b, "Force[N]", "")
    name_colunms = np.char.replace(name_colunms, "::", "_")
    name_colunms = np.char.replace(name_colunms, "-", "_")
    #clear name of colums
    left_L =[]
    right_R = []
    for i in range(len(name_colunms)):
        left_L.append(name_colunms[i].rfind("_L"))
        right_R.append(name_colunms[i].rfind("_R"))
    for j in range(len(name_colunms)):
        if left_L[j] > 0:
            name_colunms[j] = name_colunms[j][left_L[j]+1:]
        elif right_R[j]>0:
            name_colunms[j] = name_colunms[j][right_R[j]+1:]
    name_colunms = np.char.replace(name_colunms, "L ", "L")
    # make name of dataframe
    name = line1[-7:-4]
    person_surface_number = line2[8:].split("_")
    main_name = (person_surface_number[1] + "_" +
                 person_surface_number[0]+ "_" +
                 person_surface_number[2]+ "_" +
                 name)
    #read row 5 to end of data
    df = pd.read_csv(path,
                     skiprows=4,
                     sep=r"\s+",
                     decimal=",",
                     header=None)
    # rename columns of dataframe
    df.columns = name_colunms.tolist()
    df = df.rename(columns={"L":"Total_L_Foot", "R":"Total_R_Foot"})
    # drop and make time
    Time = np.arange(len(df)) * 0.01
    df = df.drop(columns="Time[secs]")
    df.insert(loc=0, column='Time', value=Time)
    #raplace -1 to 0
    df = df.replace(-1, 0)
    # make columns missed
    if len(df.columns)== 7:
        df.insert(loc=2, column= "L_Midfoot", value=0)
        df.insert(loc=6, column= "R_Midfoot", value=0)

    return name, person_surface_number, name_colunms, main_name , df

# q ,w,  s , z, x= make_dataframe(file_path)
# globals()[z] = x
# globals()[z]

# make_dataframe(file_path)
name ,person_surface_number, name_colunms, main_name, data_frame= make_dataframe(file_path)

data_frame

,Time,L_Heel,L_Midfoot,L_Forefoot,Total_L_Foot,R_Forefoot,R_Midfoot,R_Heel,Total_R_Foot
0,0.00,160.65,0,630.00,790.65,2.50,0,0.00,2.50
1,0.01,148.05,0,642.60,790.65,0.00,0,0.00,0.00
2,0.02,129.15,0,661.50,790.65,0.00,0,0.00,0.00
3,0.03,113.40,0,677.25,790.65,0.00,0,0.00,0.00
4,0.04,94.50,0,686.70,781.20,0.00,0,0.00,0.00
...,...,...,...,...,...,...,...,...,...
46886,468.86,13.65,0,581.49,595.14,74.37,0,137.35,211.72
46887,468.87,5.46,0,570.57,576.03,82.41,0,139.36,221.77
46888,468.88,0.00,0,554.19,554.19,87.10,0,147.40,234.50
46889,468.89,0.00,0,540.54,540.54,97.15,0,157.45,254.60


Read all data - Force data Foot

In [24]:
# read data form
input_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main"
# save datafrom to folder as xlsx
output_path = r"C:/Users/user/Desktop/Fatemeh/Force data (novel loadsol 2)/Clear data xlsx"

os.makedirs(input_path, exist_ok=True)
file_names = [f for f in os.listdir(input_path) if f.endswith('.txt')]

# test read data
# file_names[0]
# path = f"{input_path}/{file_names[0]}"
# q ,w,  s , z, x= make_dataframe(path)

if 'name' in globals():
    del name
if 'person_surface_number' in globals():
    del person_surface_number
if 'person_surface_number' in globals():
    del name_colunms
if 'main_name' in globals():
    del main_name
if 'data_frame' in globals():
    del data_frame

name_of_dataframe = []
for file_name in file_names:
    path = f"{input_path}/{file_name}"
    print(path)
    name ,person_surface_number, name_colunms, main_name, data_frame= make_dataframe(path)
    print(main_name)
    name_of_dataframe.append(main_name)
    globals()[main_name] = data_frame


C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-24-19-079.txt
Gravel_P02_01_079
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-38-11-074.txt
Asphalt_P02_01_074
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 09-52-01-474.txt
Sand_P02_01_474
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 10-06-34-305.txt
Grass_P02_01_305
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 10-09-51-519.txt
Grass_P02_02_519
C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2) Main/loadsolASCII_25-11-11 

In [25]:
Asphalt_S07_01_341

,Time,L_Heel,L_Midfoot,L_Forefoot,Total_L_Foot,R_Forefoot,R_Midfoot,R_Heel,Total_R_Foot
0,0.00,160.65,0,630.00,790.65,2.50,0,0.00,2.50
1,0.01,148.05,0,642.60,790.65,0.00,0,0.00,0.00
2,0.02,129.15,0,661.50,790.65,0.00,0,0.00,0.00
3,0.03,113.40,0,677.25,790.65,0.00,0,0.00,0.00
4,0.04,94.50,0,686.70,781.20,0.00,0,0.00,0.00
...,...,...,...,...,...,...,...,...,...
46886,468.86,13.65,0,581.49,595.14,74.37,0,137.35,211.72
46887,468.87,5.46,0,570.57,576.03,82.41,0,139.36,221.77
46888,468.88,0.00,0,554.19,554.19,87.10,0,147.40,234.50
46889,468.89,0.00,0,540.54,540.54,97.15,0,157.45,254.60
